In [5]:
import pandas as pd
root = r"/Users/saurabhlevin/Deployment/IDS-DRR-Assam-Risk-Model"
# Load the CSV file
file_path = root+r"/RiskScoreModel/data/Assam_data.csv"
print("Loading CSV file from:", file_path)
df = pd.read_csv(file_path)
print("CSV file loaded successfully. Shape:", df.shape)

Loading CSV file from: /Users/saurabhlevin/Deployment/IDS-DRR-Assam-Risk-Model/RiskScoreModel/data/Assam_data.csv
CSV file loaded successfully. Shape: (13330, 86)


In [6]:
# Selecting only numeric columns along with 'object-id', 'timeperiod', and 'financial-year'
numeric_columns = df.select_dtypes(include=["number"]).columns
list(numeric_columns).remove("rc-area")
numeric_columns = numeric_columns.drop("rc-area")
df_numeric = df[["object-id", "timeperiod", "financial-year"] + list(numeric_columns)]

# Identifying columns that are not included
excluded_columns = [col for col in df.columns if col not in df_numeric.columns]
print("Excluded columns:", excluded_columns)

df_numeric.head()

Excluded columns: ['district', 'rc-area', 'revenue-ci', 'dtname']


,object-id,timeperiod,financial-year,total-tender-awarded-value,erosion-tenders-awarded-value,sdrf-sanctions-awarded-value,sdrf-tenders-awarded-value,restoration-measures-tenders-awarded-value,immediate-measures-tenders-awarded-value,others-tenders-awarded-value,...,total-tender-awarded-value-fy-cumsum,sdrf-sanctions-awarded-value-fy-cumsum,sdrf-tenders-awarded-value-fy-cumsum,preparedness-measures-tenders-awarded-value-fy-cumsum,immediate-measures-tenders-awarded-value-fy-cumsum,others-tenders-awarded-value-fy-cumsum,topsis-score,risk-score,total-infrastructure-damage,total-female-population
0,18-758-00257,2021_04,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.89,5,0,337.26
1,18-756-00249,2021_04,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.75,4,0,366.24
2,18-303-00121,2021_04,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.75,4,0,306.97
3,18-320-00206,2021_04,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.74,4,0,110.49
4,18-316-00185,2021_04,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.73,4,0,692.77


In [7]:
df_melted = df_numeric.melt(id_vars=["object-id", "timeperiod", "financial-year"], var_name="factor", value_name="score")
print("Data melted. Shape:", df_melted.shape)

df_melted.head()

Data melted. Shape: (1053070, 5)


,object-id,timeperiod,financial-year,factor,score
0,18-758-00257,2021_04,2021-2022,total-tender-awarded-value,0.0
1,18-756-00249,2021_04,2021-2022,total-tender-awarded-value,0.0
2,18-303-00121,2021_04,2021-2022,total-tender-awarded-value,0.0
3,18-320-00206,2021_04,2021-2022,total-tender-awarded-value,0.0
4,18-316-00185,2021_04,2021-2022,total-tender-awarded-value,0.0


In [8]:

df_transposed = df_melted.pivot(index=["factor", "timeperiod", "financial-year"], columns="object-id", values="score").reset_index()
print("Data pivoted. Shape:", df_transposed.shape)
df_transposed.head()
# Save or display the transformed data
output_file = "Transformed_Assam_Data.csv"
df_transposed.to_csv(output_file, index=False)
print("Transformed data saved to:", output_file)

Data pivoted. Shape: (4898, 218)
Transformed data saved to: Transformed_Assam_Data.csv


In [9]:
# Verify with the source data. Given factor, district, timeperiod and object-id, the score should match

factor = "sdrf-sanctions-awarded-value"
district = '18-319-00202'
timeperiod = '2025_07'
df_transposed.head()
print("modified")
print(df_transposed[(df_transposed["factor"] == factor) & (df_transposed["timeperiod"] == timeperiod) ][[district, "timeperiod", "financial-year", "factor"]])
print("original")
df[(df["object-id"] == district) & (df["timeperiod"] == timeperiod)][['timeperiod', 'financial-year', factor]]



modified
object-id  18-319-00202 timeperiod financial-year  \
3833                0.0    2025_07      2025-2026   

object-id                        factor  
3833       sdrf-sanctions-awarded-value  
original


,timeperiod,financial-year,sdrf-sanctions-awarded-value
9329,2025_07,2025-2026,0.0


In [6]:
dff = pd.read_csv(root+r"/RiskScoreModel/data/Transformed_Assam_Data.csv")
conditions = []
operator_map = {
            '==': lambda col, val: col == val,
            '!=': lambda col, val: col != val,
            '>': lambda col, val: col > val,
            '<': lambda col, val: col < val,
            '>=': lambda col, val: col >= val,
            '<=': lambda col, val: col <= val,
            'in': lambda col, val: col.isin(val),
            'not in': lambda col, val: ~col.isin(val)
        }
conditions.append(operator_map["=="](dff["factor"], "sdrf-sanctions-awarded-value"))

dff = dff[pd.concat(conditions, axis=1).all(axis=1)]
dff

,factor,timeperiod,financial-year,18-300,18-300-00101,18-300-00102,18-300-00103,18-300-00104,18-300-00105,18-300-00106,...,18-799,18-799-00124,18-799-00125,18-799-00265,18-816,18-816-00235,18-816-00236,18-816-00258,18-816-00259,18-816-00261
3850,sdrf-sanctions-awarded-value,2021_04,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3851,sdrf-sanctions-awarded-value,2021_05,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3852,sdrf-sanctions-awarded-value,2021_06,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3853,sdrf-sanctions-awarded-value,2021_07,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3854,sdrf-sanctions-awarded-value,2021_08,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3855,sdrf-sanctions-awarded-value,2021_09,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3856,sdrf-sanctions-awarded-value,2021_10,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3857,sdrf-sanctions-awarded-value,2021_11,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3858,sdrf-sanctions-awarded-value,2021_12,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3859,sdrf-sanctions-awarded-value,2022_01,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
